# Rotated surface-code gadget explorer

Select a physical DEQ gadget to inspect its Stim circuit in Crumble. The visualization uses the same explicit reset/CNOT/measurement bodies emitted into the generated `.deq` source.

For a single d=3 patch, the layout is read directly from Stim's `surface_code:rotated_memory_z` circuit and scaled by 1/2: data qubits lie at half-integer coordinates, and syndrome ancillas lie at integer coordinates. In this orientation the top and bottom boundary checks are X type; the left and right boundary checks are Z type.

The **PrepareYBoundary** entries show the first stage of the in-place logical-|+i⟩ protocol: a missing-corner, full-rank XXZZ boundary patch followed by repeated stabilizer extraction. **YBoundaryToRotated** then runs the reverse-time diagonal-twist circuit that completes the preparation.

In [3]:
import importlib.util
from pathlib import Path

import ipywidgets as widgets
from IPython.display import HTML, display


def find_repository_root() -> Path:
    for directory in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
        if (directory / 'tools' / 'generate_rotated_surface_code_deq.py').is_file():
            return directory
    raise RuntimeError('Could not find tools/generate_rotated_surface_code_deq.py')


root = find_repository_root()
generator_path = root / 'tools' / 'generate_rotated_surface_code_deq.py'
spec = importlib.util.spec_from_file_location('rotated_surface_code_deq', generator_path)
generator = importlib.util.module_from_spec(spec)
assert spec.loader is not None
spec.loader.exec_module(generator)
from surface_code_deq.gadget_visualizations import gadget_circuits
from surface_code_deq.prepare_y_gadgets import y_boundary_circuits

In [4]:
gadget_descriptions = {
    'PrepareX': 'Prepare a logical |+> patch, then perform one syndrome-extraction round.',
    'PrepareZ': 'Prepare a logical |0> patch, then perform one syndrome-extraction round.',
    'PrepareYBoundary': 'First Prepare-Y stage: initialize the missing-corner, full-rank XXZZ boundary patch and project its stabilizers.',
    'YBoundarySyndromeExtraction': 'One repeated extraction round on the full-rank XXZZ boundary patch; run floor(d/2) additional rounds after preparation, for ceil(d/2) total XXZZ rounds.',
    'YBoundaryToRotated': 'Reverse-time diagonal-twist circuit: restore the missing corner in |+i⟩ and turn the XXZZ boundary state into a standard rotated surface-code patch. The final classical frame updates are present in the emitted DEQ source but omitted here because Crumble renders record controls as ERR markers.',
    'SyndromeExtraction': 'One complete X-check then Z-check extraction round.',
    'MeasureX': 'Destructively measure the data patch in the X basis.',
    'MeasureZ': 'Destructively measure the data patch in the Z basis.',
    'TransversalHadamard': 'Apply H to every data qubit, producing the time-like domain-wall frame.',
    'HadamardExtend': 'Our Fig. 2(c) transform: keep the exact H-conjugated RotatedSurfaceCode input on the left, prepare d²−1 right-side |+> sites, use Z checks along the full bottom and right boundaries, and measure the local CSS checks in four CNOT layers.',
    'HadamardExtensionSE': 'Repeat the four-layer local CSS syndrome round on the 2d²−1-site extension patch.',
    'HadamardCornerMove': 'Reset the missing bottom-right site, replace the bottom Z half-checks by stagger-shifted X half-checks, and retain the d-row by 2d-column layout.',
    'HadamardDomainWallSE': 'Repeat the four-layer CSS round with the corner-moved bottom X boundary.',
    'HadamardShrink': 'Z-measure the left H-frame patch and extract the final syndrome round on the retained right patch.',
    'HadamardReturnExtend': 'Reinitialize d(d-1) sites to the left and extend into a d-row by (2d-1)-column patch.',
    'HadamardReturnExtensionSE': 'Repeat one standard syndrome round on the horizontal return-extension patch.',
    'HadamardReturnShrink': 'Z-measure the right d(d-1) sites and retain the left square patch.',
    'HadamardSwapQECNW': 'Apply parallel native SWAP gates northwest, then extract one syndrome round.',
    'HadamardSwapQECSW': 'Apply parallel native SWAP gates southwest, then extract one syndrome round, moving one data column left overall.',
    'MergeBeginXX': 'Horizontal MXX: prepare the seam in |0> and join facing Z boundaries.',
    'MergedSEXX': 'One complete extraction round on the horizontally merged patch.',
    'MergeEndXX': 'Horizontal MXX: measure the seam in Z and split the patches.',
    'MergeBeginZZ': 'Vertical MZZ: prepare the seam in |+> and join facing X boundaries.',
    'MergedSEZZ': 'One complete extraction round on the vertically merged patch.',
    'MergeEndZZ': 'Vertical MZZ: measure the seam in X and split the patches.',
}

distance = widgets.Dropdown(options=(3, 5, 7), value=3, description='Distance:')
gadget = widgets.Dropdown(
    options=tuple(gadget_descriptions),
    value='HadamardExtend',
    description='Gadget:',
)
crumble_output = widgets.Output()


def render_gadget(_=None):
    selected = gadget.value
    basis = 'ZZ' if selected.endswith('ZZ') else 'XX'
    circuits = (
        y_boundary_circuits(distance.value)
        if selected.startswith('YBoundary') or selected == 'PrepareYBoundary'
        else gadget_circuits(distance.value, basis)
    )
    circuit = circuits[selected]
    crumble_html = circuit.diagram('interactive-html')._repr_html_()
    # Crumble's 300 px default compresses its canvas and timeline.
    crumble_html = crumble_html.replace('height: 300px;', 'height: 850px;')

    with crumble_output:
        crumble_output.clear_output(wait=False)
        display(HTML(f'<h2>{selected}</h2>'))
        display(HTML(f'<p>{gadget_descriptions[selected]}</p>'))
        display(HTML(crumble_html))
        display(HTML(
            f'<p><a href="{circuit.to_crumble_url()}" target="_blank" rel="noopener">'
            'Open this circuit in a full Crumble tab</a></p>'
        ))


distance.observe(render_gadget, names='value')
gadget.observe(render_gadget, names='value')
display(widgets.VBox([widgets.HBox([distance, gadget]), crumble_output]))
render_gadget()

## Coordinate convention

The display coordinate system is intentionally separate from DEQ’s qubit IDs. For the single-patch gadgets it is an exact half-scale copy of Stim's coordinates: data positions are `(x + 0.5, y + 0.5)`, interior check ancillas sit at the centers of their four data qubits, and two-body boundary checks lie half a cell beyond the patch. This keeps data and ancillas on alternating lattice sites in Crumble. In the Hadamard extension views, the original patch is the left square of a `d`-row by `2d`-column layout. Fig. 2(d), transformed into our convention, replaces the bottom Z half-checks with X half-checks on the complementary stagger.